In [ ]:
# Private UROP ViT-B/16; immutable source and invocation settings.
SETTINGS = {'mode': 'train', 'world': 2, 'quota_used_minutes': 712.0, 'accounting_multiplier': 1, 'stop_after_seconds': None, 'recovery': 'auto', 'expected_recovery_sha256': None}
PAYLOAD = (
    '...'
)
SESSIONS_SHA = '6438d30a6aa07c4d9edf45495292dce75b5c562708834ee4f8afc455475875c0'
# Verify the paused training state before the controller copies any input.
from pathlib import Path
import hashlib
import json

matches = [
    path for path in Path('/kaggle/input').glob('**/urop_vit_finetune/sessions.json')
    if hashlib.sha256(path.read_bytes()).hexdigest() == SESSIONS_SHA
]
if len(matches) != 1:
    raise RuntimeError('Mount the checkpointed training version identified by its session receipt')
recovery = matches[0].parent
sessions = json.loads(matches[0].read_text())
if (sessions[-1]['state'] != 'paused' or not (recovery/'registration.json').is_file()
        or not any(recovery.glob('*/resume.pt'))):
    raise RuntimeError('Mounted output is missing the paused training registration or checkpoints')
SETTINGS['recovery'] = str(recovery)
from pathlib import Path
import base64, hashlib, json, sys, zlib
source = Path('/tmp/urop_vit_finetune_source')
source.mkdir(exist_ok=True)
for name, (digest, body) in json.loads(zlib.decompress(base64.b64decode(PAYLOAD))).items():
    raw=base64.b64decode(body)
    if hashlib.sha256(raw).hexdigest()!=digest:
        raise RuntimeError('Embedded source hash changed: '+name)
    (source/name).write_bytes(raw)
sys.path.insert(0, str(source))
from vit_finetune_cloud_v1 import start
start(source, Path('/kaggle/working/urop_vit_finetune'), **SETTINGS)
